In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for updating sales KPI logic and output in LifeScience Sales Pipeline
# Purpose: Update bonus eligibility, performance_flag, product_perf_band, and add rep_tier per new requirements
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script updates the logic for bonus eligibility, performance_flag, product_perf_band, and adds rep_tier.
#              The final output displays the enriched KPI DataFrame and rep summary with all updated columns.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql.functions import col, when, sum, avg, max, count, countDistinct, lag, rank, round, to_date, year, month, quarter, stddev, min, current_date, months_between  
from pyspark.sql.types import DoubleType, StringType, LongType  
from pyspark.sql.window import Window  

# PART 1: READ THE DATA (PRODUCT, REP, MARKET, SALES, REGION, CHANNELS)
product_df = spark.read.table("purgo_playground.product_data")
rep_df = spark.read.table("purgo_playground.rep_data")
market_df = spark.read.table("purgo_playground.market_data")
channel_df = spark.read.table("purgo_playground.channel_data")
incentive_df = spark.read.table("purgo_playground.incentive_data")
sales_df = spark.read.table("purgo_playground.sales_transaction_data")

# PART 2 - DATE ENRICHMENTS
sales_df = sales_df.withColumn("sales_date", to_date("sales_date")) \
                   .withColumn("year", year("sales_date")) \
                   .withColumn("month", month("sales_date")) \
                   .withColumn("quarter", quarter("sales_date"))

# PART 3 - DATA ENRICHMENT (JOIN ALL DIMENSIONS)
enriched_df = sales_df \
    .join(product_df, on="product_id", how="left") \
    .join(rep_df, on="rep_id", how="left") \
    .join(market_df, on="product_id", how="left") \
    .join(channel_df, on="product_id", how="left") \
    .join(incentive_df, on="rep_id", how="left")

def update_kpi_logic(df):
    """
    Updates KPI logic for bonus_eligibility, performance_flag, product_perf_band, and adds rep_tier.
    Args:
        df (DataFrame): The enriched sales DataFrame.
    Returns:
        Tuple[DataFrame, DataFrame]: (kpi_df, rep_summary) with updated columns.
    """
    # Update incentive_per_sale calculation
    kpi_df = df.withColumn("incentive_per_sale", col("incentive_amount") / col("sales_amount"))
    # Update performance_flag logic per new requirement
    kpi_df = kpi_df.withColumn("performance_flag",
        when(col("sales_amount") > 9000, "High")
        .when((col("sales_amount") <= 9000) & (col("sales_amount") > 7000), "Medium")
        .otherwise("Low")
    )
    # PART 6 - WINDOW METRICS
    window_spec = Window.partitionBy("rep_id").orderBy("sales_date")
    kpi_df = kpi_df.withColumn("rep_running_total", sum("sales_amount").over(window_spec)) \
                   .withColumn("prev_sales", lag("sales_amount").over(window_spec)) \
                   .withColumn("sales_growth", round((col("sales_amount") - col("prev_sales")) / col("prev_sales"), 2))
    # PART 7 - ROLLING AVERAGE & QUARTERLY METRICS
    rolling_spec = Window.partitionBy("rep_id").rowsBetween(-2, 0)
    kpi_df = kpi_df.withColumn("rolling_avg_sales", round(avg("sales_amount").over(rolling_spec), 2))
    # PART 8 - SALES BUCKET TAGGING
    kpi_df = kpi_df.withColumn("sales_bucket",
        when(col("sales_amount") > 13000, ">13K")
        .when((col("sales_amount") > 9000), "9K-13K")
        .when((col("sales_amount") > 7000), "7K-9K")
        .otherwise("<7K")
    )
    # PART 9 - TOP PERFORMERS RANKING
    ranking_spec = Window.partitionBy("year").orderBy(col("sales_amount").desc())
    kpi_df = kpi_df.withColumn("annual_rank", rank().over(ranking_spec))
    # PART 10 - CAGR CALCULATION (YEARLY SALES)
    yearly_sales = kpi_df.groupBy("year").agg(sum("sales_amount").alias("total_year_sales"))
    # PART 11 - REP SUMMARY & BONUS ELIGIBILITY
    # Update bonus_eligibility logic: 'Yes' if total_rep_sales > 25000
    rep_summary = kpi_df.groupBy("rep_id", "rep_name").agg(
        count("transaction_id").alias("txn_count"),
        sum("sales_amount").alias("total_rep_sales"),
        avg("sales_amount").alias("avg_rep_sales"),
        max("sales_amount").alias("max_rep_sale")
    ).withColumn("bonus_eligibility", when(col("total_rep_sales") > 25000, "Yes").otherwise("No"))
    # Add rep_tier column per new requirement
    rep_summary = rep_summary.withColumn("rep_tier",
        when(col("total_rep_sales") > 45000, "Platinum Plus")
        .when(col("total_rep_sales") > 35000, "Platinum")
        .when(col("total_rep_sales") > 25000, "Gold")
        .otherwise("Silver")
    )
    # PART 12 - ADVANCED TRANSFORMATIONS (ADDITIONAL INSIGHTS)
    # Update product_perf_band logic per new requirement
    kpi_df = kpi_df.withColumn("product_perf_band",
        when(col("sales_amount") > 10000, "Excellent")
        .when(col("sales_amount") > 8000, "Good")
        .when(col("sales_amount") > 5000, "Moderate")
        .otherwise("Poor")
    )
    # Update bonus_eligibility at transaction level: 'Yes' if sales_amount > 25000, else previous logic
    kpi_df = kpi_df.withColumn("bonus_eligibility",
        when(col("sales_amount") > 25000, "Yes").otherwise("No")
    )
    return kpi_df, rep_summary

# Apply updated KPI logic
kpi_df, rep_summary = update_kpi_logic(enriched_df)

# Display the Output
print("===== KPI Enriched Data Preview =====")
display(kpi_df.select(
    "transaction_id", "rep_name", "product_name", "sales_amount", "sales_bucket",
    "performance_flag", "rep_running_total", "sales_growth", "rolling_avg_sales",
    "annual_rank", "product_perf_band"
))

print("===== Rep Summary and Bonus Eligibility =====")
display(rep_summary.select(
    "rep_id", "rep_name", "txn_count", "total_rep_sales", "avg_rep_sales",
    "max_rep_sale", "bonus_eligibility", "rep_tier"
))

# spark.stop()  # Do not stop SparkSession in Databricks
